# 환경설정

In [49]:
import os
from neo4j import GraphDatabase
from langchain.chains import RetrievalQA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field
from typing import List
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.graphs.neo4j_graph import Neo4jGraph
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores.neo4j_vector import remove_lucene_chars
from tqdm import tqdm
from modules import logging
from jina import Client
from langchain_community.vectorstores import FAISS
import numpy as np
import ollama
from langchain_community.embeddings import OllamaEmbeddings

In [2]:
logging.langsmith("Model_RAG")

LangSmith 추적을 시작합니다.
[프로젝트명]
Model_RAG


In [3]:
# API KEY를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API KEY 정보로드
load_dotenv()

True

In [4]:
def show_metadata(docs):
    if docs:
        print("[metadata]")
        print(list(docs[0].metadata.keys()))
        print("\n[examples]")
        max_key_length = max(len(k) for k in docs[0].metadata.keys())
        for k, v in docs[0].metadata.items():
            print(f"{k:<{max_key_length}} : {v}")

In [5]:
FILE_PATH = "data\무배당 메리츠 다이렉트 운전자보험2409.pdf"
raw_documents = PyPDFLoader(FILE_PATH).load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=125)
documents = text_splitter.split_documents(raw_documents)

In [6]:
print(f"분할된 청크의수: {len(documents)}")

분할된 청크의수: 54


In [50]:
ollama_embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

In [51]:
vectorstore = FAISS.from_documents(documents=documents, embedding=ollama_embeddings)

In [52]:
vectorstore.save_local('./db/faiss')

In [ ]:
# vectorstore = FAISS.load_local('./db/faiss', ollama_embeddings)

In [ ]:
llm = ChatOpenAI(model_name="gpt-3.5-turbo-0125", temperature=0)